In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn import datasets, svm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd

# Lab 12: Gaussian Processes and Bayesian Optimization

The goal of this lab is to build a simple autoML method for choosing the optimal hyperparameter of a XXX algorithm. It is organized in 3 parts: 
1. Implementing a Gaussian process
2. Implementing a simple BO using UCB
3. Application to hyperparameter tuning

## Part 1: Gaussian process

A Gaussian process $f(x) \sim \mathcal{GP}(m(x), K(x, x^\prime))$ is a (potentially infinite) collection of random variables, any finite subsample of which has a joint Gaussian distribution. If $(X_1, \dotsc, X_N)$ is a subsample of $f(x)$, we have 
$$
    (f_1, \dotsc, f_N) \sim \mathcal{N}\left(\begin{pmatrix} m(\mathbf{x}_{1}) \\ \vdots \\ m(\mathbf{x}_{N}) \end{pmatrix}, \begin{pmatrix} K(\mathbf{x}_{1}, \mathbf{x}_{1}) & \dotsc & K(\mathbf{x}_{1}, \mathbf{x}_{N}) \\ \vdots & \ddots & \vdots \\ K(\mathbf{x}_{N}, \mathbf{x}_{1}) & \cdots & K(\mathbf{x}_{N}, \mathbf{x}_{N}) \end{pmatrix}  \right)
$$
where $f_i = f(x_i)$.

A Gaussian process can be interpreted as a distribution over functions. The choice of the kernel $K(x, x^\prime)$ imposes some conditions on the function. An example of a kernel is the Squared Exponential (SE) kernel, also called the Radial Basis Function (RBF).  

In [ ]:
def squared_exponential_kernel(x1, x2, length_scale=1.0, variance=1.0):
    """
    Squared Exponential (RBF) kernel function.

    Parameters:
        x1, x2 : numpy arrays
            Input points for the kernel function.
        length_scale : float
            Controls the smoothness of the function.
        variance : float
            Vertical variation (amplitude).

    Returns:
        Kernel matrix between x1 and x2.
    """
    sqdist = x1**2 + x2.T**2 - 2 * (x1 @ x2.T)
    return variance * np.exp(-0.5 * sqdist / length_scale**2)

### Representation of a Gaussian process

We usually represent a Gaussian process by plotting its mean, namely the function $x \mapsto \mathbb{E}[f(x)]$ for all $x$, as well as a confidence interval around $x$. The following function plots a Gaussian process and will be used throughout this notebook.

In [ ]:
def plot_GP(x, mu_gp, lower_confidence, upper_confidence):
    plt.figure(figsize=(10, 6))
    plt.plot(x, mu_gp, "b", label="Predictive Mean")
    plt.fill_between(x.ravel(),
                     lower_confidence,
                     upper_confidence,
                     color="blue", alpha=0.2, label="95% Confidence Interval")

    plt.title("Representation of the Gaussian Process")
    plt.xlabel("x")
    plt.ylabel("f(x)")
    plt.legend()
    plt.show()

Now, let's define the domain and generate the corresponding RBF kernel.

In [ ]:
n_samples = 100

# Define the input space
x = np.linspace(-5, 5, n_samples).reshape(-1, 1)  # 100 points in [-5, 5]

# Mean of the Gaussian Process
mean = np.zeros(n_samples,) # To be replaced with the function of your choice if you don't want a 0 mean

# Kernel matrix
length_scale = 1.0
variance = 1.0
kernel_matrix = squared_exponential_kernel(x, x, length_scale, variance)

# For fun: let's plot the kernel matrix
plt.figure()
plt.imshow(kernel_matrix)
plt.show()

#### Question 1
Given $x$, what is the distribution of $f(x)$? What is $\mathbb{E}[f(x)]$? What is the 95% confidence interval around $\mathbb{E}[f(x)]$? Implement these values.

*(Hint: A 95% confidence interval corresponds to a [z-score](https://en.wikipedia.org/wiki/Standard_score) of 1.96)*

In [ ]:
mu_gp = # Your code here
upper_confidence = # Your code here
lower_confidence = # Your code here

# Plot the results
plot_GP(x, mu_gp, lower_confidence, upper_confidence)

### Sampling from a Gaussian process

Seeing a GP as a distribution over functions, an important task is to sample from the GP. This corresponds to sampling functions on the domain space. Such functions will not have analytical expressions, and consequently we need to evaluate the sampled functions on a set of points from the domain space. 

#### Question 2

Given a vector $(x_1, \dotsc, x_N)$, how can we sample $(f(x_1), \dotsc, f(x_N))$ from the GP? Implement this method.

In [ ]:
def sample_GP(mean_func, kernel_func, x):
    f = # Your code here
    return f

mean_func = lambda x: 0
kernel_func = squared_exponential_kernel

f = sample_GP(mean_func, kernel_func, x)

plt.figure(figsize=(10, 6))
plt.plot(x, f, linestyle="--")
plt.title("Sampling from Gaussian Process")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.show()

### Other kernels

Until now, we used only the RBF kernel. Other kernels can be used in practice, that will induce various properties on the distribution over functions. 

#### Question 3 (*)

In addition to the already implemented Matérn kernel, implement the linear kernel $K(x, x^\prime) = x^T \Sigma x^\prime$ and the Exponential Sine Squared Kernel. 

In [ ]:
def matern_kernel(x1, x2, length_scale=1.0, variance=1.0, nu=1.5, epsilon=1e-6):
    """
    Matérn kernel function.

    Parameters:
        x1, x2 : numpy arrays
            Input points for the kernel function.
        length_scale : float
            Controls the smoothness of the function.
        variance : float
            Vertical variation (amplitude).
        nu : float
            Smoothness parameter.

    Returns:
        Kernel matrix between x1 and x2.
    """
    dists = np.sqrt(np.sum(x1**2, axis=1).reshape(-1, 1) + np.sum(x2**2, axis=1) - 2 * np.dot(x1, x2.T))
    dists = np.maximum(dists, epsilon)  # Avoid zero distances

    if nu == 0.5:
        # Special case: Exponential kernel
        return variance * np.exp(-dists / length_scale)
    else:
        factor = (2**(1 - nu)) / gamma(nu)
        scaled_dists = np.sqrt(2 * nu) * dists / length_scale
        bessel_term = kv(nu, scaled_dists)
        bessel_term[~np.isfinite(bessel_term)] = 0  # Handle NaN or Inf in Bessel function
        return variance * factor * (scaled_dists**nu) * bessel_term
    
def linear_kernel(x1, x2, Sigma):
    return None # Your code here

def exp_sine_squared_kernel(x1, x2, length_scale=1.0, variance=1.0, periodicity=1.0):
    return None # Your code here

### Introducing observations

When observations are available, we must split the outputs of the GP between observed outputs for inputs $\mathbf{X}$, written $\mathbf{f}$ and the predicted outputs for inputs $\mathbf{X}_*$, written $\mathbf{f}_*$. Using the notation $K(\mathbf{X}, \mathbf{X}^\prime)$ for the matrix of elements $K(x_i, x^\prime_j)$, we have then:
$$
\begin{pmatrix} \mathbf{f} \\ \mathbf{f}_* \end{pmatrix}
 \sim \mathcal{N}\left(\begin{pmatrix} m(\mathbf{X}) \\ m(\mathbf{X_*}) \end{pmatrix} , \begin{pmatrix} K(\mathbf{X}, \mathbf{X}) & K(\mathbf{X}, \mathbf{X}_*) \\ K(\mathbf{X}_*, \mathbf{X})  & K(\mathbf{X}_*, \mathbf{X}_*) \end{pmatrix}  \right)
$$

#### Question 4 (*)

Using the results presented in slide 7, what is the distribution of $p(\mathbf{f}_* | \mathbf{f}, \mathbf{X}, \mathbf{X}_*)$? Implement a visualization of this posterior as in Question 1. You can test it with assuming that the GP has a mean equal to 0, and with the kernel of your choice. 

In [ ]:
def posterior_noise_free(mean_func, kernel_func, x, X_train, f_train):
    mu_gp = # Your code here
    upper_confidence = # Your code here
    lower_confidence = # Your code here
    return mu_gp, lower_confidence, upper_confidence

x_train = np.array([-4, -3, -2, 0, 2]).reshape(-1, 1)  # Training points
f_train = np.sin(x_train)  # Training targets

x = np.linspace(-5, 5, n_samples).reshape(-1, 1)  # 100 points in [-5, 5]


mu_gp, lower_confidence, upper_confidence = posterior_noise_free(mean_func, kernel_func, x, X_train, f_train)

# Plot the results
plot_GP(x, mu_gp, lower_confidence, upper_confidence)

In practice, we rarely observe the exact values $\mathbf{f}$, but rather noisy observations. Assuming a normal noise: $y \sim \mathcal{N}(f, \sigma^2)$, we can modify the GP as follows:
$$
\begin{pmatrix} \mathbf{y} \\ \mathbf{f}_* \end{pmatrix}
 \sim \mathcal{N}\left(\begin{pmatrix} m(\mathbf{X}) \\ m(\mathbf{X_*}) \end{pmatrix} , \begin{pmatrix} K(\mathbf{X}, \mathbf{X}) + \sigma^2 \mathbf{I} & K(\mathbf{X}, \mathbf{X}_*) \\ K(\mathbf{X}_*, \mathbf{X})  & K(\mathbf{X}_*, \mathbf{X}_*) \end{pmatrix}  \right)
$$

#### Question 5 (*)

Propose an adaptation of your answer to question 4 that integrates this modification.

In [ ]:
def posterior_noisy(mean_func, kernel_func, x, X_train, y_train, sigma2):
    mu_gp = # Your code here
    upper_confidence = # Your code here
    lower_confidence = # Your code here
    return mu_gp, lower_confidence, upper_confidence

sigma2 = .1

x_train = np.array([-4, -3, -2, 0, 2]).reshape(-1, 1)  # Training points
f_train = np.sin(x_train)  # Training targets
y_train = f_train + np.random.normal(0, noise_std, y_train.shape)

x = np.linspace(-5, 5, n_samples).reshape(-1, 1)  # 100 points in [-5, 5]


mu_gp, lower_confidence, upper_confidence = posterior_noise_free(mean_func, kernel_func, x, X_train, y_train, sigma2)

# Plot the results
plot_GP(x, mu_gp, lower_confidence, upper_confidence)

## Part 2: Bayesian Optimization

Bayesian Optimization aims to find the optimum of a black-box function by iteratively sampling from it. BO relies on two components: a GP to represent the current state of belief about the function to optimize, and an acquisition function, the goal of which is to determine from which point to sample next. 

For the GP update, we will use the code proposed in Question 5 (or Question 4 if you didn't have time to solve Question 5).

For the acquisition function, we will use the UCB criterion:

$$
\alpha_{\text{UCB}}(x; \lambda) = \mu(x) + \lambda \sigma(x)
$$

where $\mu(x)$ is the expected mean of the GP and $\sigma(x)$ the expected standard deviation of the GP. When aiming to maximize function $f$, we choose the next point to sample from by maximizing the UCB. 

In practice, we will discretize the input space, compute the UCB for each value, and take the maximal value. 

#### Question 6 (*)

Implement the UCB maximization and run a BO on the defined function. 

In [ ]:
def UCB(x, X_obs, y_obs, mean_func, kernel_func, sigma2):
    """
    Maximizes the UCB for the current GP.

    Parameters:
        x : numpy array
            Discretized input space
        X_obs, y_obs : numpy arrays
            Available observations.
        mean_func, kernel_func : functions
            Mean and kernel of the GP
        sigma2 : float
            Variance of the noise for the observations.

    Returns:
        Value of the point to sample from.
    """
    
    ucb = 0 # Your code here
    argmax_x_ucb = np.argmax(ucb)
    return x[argmax_x_ucb]


objective_function = lambda x: np.sin(x) - 0.05 * x**2

sigma2 = 0.1

x = np.linspace(-5, 5, n_samples).reshape(-1, 1)  # 100 points in [-5, 5]

# First step: choosing observation at random
X_obs = np.array([x[int(n_samples / 2)],])
y_obs = np.array(objective_function(X_obs[0]) + np.sqrt(sigma2) * np.random.randn())

# Running the BO

n_steps_BO = 10

for t in range(n_steps_BO):
    x_new = UCB(x, X_obs, y_obs, mean_func, kernel_func, sigma2)
    y_new = objective_function(x_new) + np.sqrt(sigma2) * np.random.randn()
    
    X_obs = np.hstack((X_obs, [x_new]))
    y_obs = np.hstack((y_obs, y_new))
    
    mu_gp, lower_confidence, upper_confidence = posterior_noise_free(mean_func, kernel_func, x, X_obs, y_obs, sigma2)
    plot_GP(x, mu_gp, lower_confidence, upper_confidence)

## Part 3: Application to autoML (optional, but easy)

All machine learning methods have hyperparameters. The choice of the hyperparameters strongly influence the quality of the trained classifier. The goal of autoML is to find the optimal hyperparameters automatically. 

In this exercise, we will consider the famous Iris dataset and will train the SVC linear classifier, as implemented in the scikit-learn library. This classifier has one parameter, $C$, which corresponds to the weight of the regularization. We will aim to find an optimal value for $C$.

The following code shows how to train a LinearSVC using scikit-learn for a specific choice of $C$.


In [ ]:
# Load iris dataset
iris = datasets.load_iris()

# Prepare X and y as dataframe
X = iris['data']
y = iris['target']

# Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

C_values = np.linspace(0.01, 5, n_samples).reshape(-1, 1)

# Training LinearSVC with C = 1
model = svm.LinearSVC(C=1, max_iter=10000)
model.fit(X_train, y_train.ravel())
score = accuracy_score(model.predict(X_test), y_test)

print('Score:', score)

#### Question 7

Modify this code to perform a grid search on $C$. What is the optimal value found and the corresponding score?

In [ ]:
def grid_search(C_values, X_train, X_test, y_train, y_test):
    optimal_C = # Your code here
    optimal_score = # Your code here
    return optimal_C, optimal_score

#### Question 8

Use Bayesian Optimization to find the optimum. Compare the found value and the optimal score for the grid search and the BO.

In [ ]:
def BO_search(C_values, X_train, X_test, y_train, y_test):
    optimal_C = # Your code here
    optimal_score = # Your code here
    return optimal_C, optimal_score